# m-out-of-n Bootstrap Scaling Experiment

Does scaling the m-out-of-n bootstrap correction by $\sqrt{m/N}$ fix overcorrection?

In [1]:
%run _dev_setup.py

🔁 Autoreload is ON (IPython detected).
✅ Using winners_curse from: /Users/iamsikun/research/winners-curse/src/winners_curse


In [2]:
from winners_curse.analysis import plot_snr_wc_comparison, plot_sample_size_wc_comparison, tabulate_sample_size_moon_wc

In [3]:
import pickle
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'serif'
tick_label_size = 12
axis_label_size = 14
legend_label_size = 12
title_size = 16

## Load Results

In [ ]:
result_dir = '../results/ab_test_moon_scaling_20260917_120221/'

with open(f'{result_dir}/results.pkl', 'rb') as f:
    results = pickle.load(f)

with open(f'{result_dir}/config.json', 'r') as f:
    config = json.load(f)

In [ ]:
results[(1000, (1, 1.005))].keys()

In [ ]:
# Estimator display names
estimator_keys = [
    'nc',
    'standard_bootstrap',
    'moon_070_scaled',
    'moon_080_scaled',
    'moon_090_scaled',
    'moon_095_scaled',
]

estimator_labels = {
    'nc': 'No Correction',
    'standard_bootstrap': 'Standard Bootstrap',
    'moon_070_scaled': r'm-out-of-n ($\gamma$=0.7)',
    'moon_080_scaled': r'm-out-of-n ($\gamma$=0.8)',
    'moon_090_scaled': r'm-out-of-n ($\gamma$=0.9)',
    'moon_095_scaled': r'm-out-of-n ($\gamma$=0.95)',
}

sample_sizes = sorted(set(k[0] for k in results.keys()))
tau_tuples = sorted(set(k[1] for k in results.keys()))
print(f'Sample sizes: {sample_sizes}')
print(f'Tau tuples: {tau_tuples}')

## SNR Comparison (Fixed Sample Size)

In [ ]:
fixed_n = 2500

# Filter results for the fixed sample size, keyed by tau tuple
snr_results = {tau: res for (n, tau), res in results.items() if n == fixed_n}

plot_snr_wc_comparison(snr_results, config)

## Sample Size Comparison (Fixed SNR)

In [ ]:
import copy

fixed_tau = tau_tuples[1]
print(fixed_tau)

# Filter results for the fixed SNR, keyed by sample size
sample_size_results = {n: res for (n, tau), res in results.items() if tau == fixed_tau}

# Build a config that drops unscaled moon estimators and renames scaled ones
plot_config_dict = copy.deepcopy(config)
plot_config_dict['estimators_dict'] = {
    k: v for k, v in plot_config_dict['estimators_dict'].items()
    if not k.endswith('_unscaled')
}
for k, v in plot_config_dict['estimators_dict'].items():
    if k.endswith('_scaled'):
        v['display_name'] = v['display_name'].replace(', scaled', '')

plot_sample_size_wc_comparison(
    sample_size_results, plot_config_dict, sample_sizes,
    save_path='moon_robustness.svg',
)

In [ ]:
moon_wc_table = tabulate_sample_size_moon_wc(sample_size_results, config, sample_sizes)
moon_wc_table.round(2)